In [ ]:
from iFinDPy import *
import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL
from datetime import date
import pandas as pd
import numpy as np

In [ ]:
THS_iFinDLogin('lzxh0011','450503')
#data=THS_DS('430198.BJ','ths_af_stock;ths_close_price_stock;ths_vol_stock;ths_market_value_stock',';100;100;','Fill:Blank','2022-05-01','2022-05-05')

In [ ]:
host='localhost'
user='Local_Editor'
password='QuantumGalaxy'
database='qgdbs'
mysql=MYSQL(host,user,password,database)

In [ ]:
THS_DS('ACU.A','ths_close_price_uss;ths_vol_uss;ths_af_uss;ths_market_value_uss','100;;;USD','Fill:Blank','2022-05-01','2022-05-05').data

In [ ]:
def get_logger():
    logger=logging.getLogger('logger')
    logger.setLevel(logging.INFO)
    fh=logging.FileHandler("E:/wangzhilin/QuantumGalaxy/logs/ticker_data_log.log",'a')
    fh.setLevel(logging.INFO)
    ch=logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter=logging.Formatter(fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    fh.setFormatter(formatter)
    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger

In [ ]:
logger=get_logger()

In [ ]:
def query_code(mysql):
    sql='select code,region from ticker_info where catagory="stock"'
    result=mysql.read_query(sql)
    cnstr=hkstr=usstr=''
    for code,region in result:
        if region=='CN':
            cnstr+=(code+',')
        elif region=='HK':
            hkstr+=(code+',')
        elif region=='US':
            usstr+=(code+',')
    cnstr=cnstr[:-1]
    hkstr=hkstr[:-1]
    usstr=usstr[:-1]
    result=[cnstr,hkstr,usstr]
    return result

In [ ]:
sql_list=['select code from ticker_info where catagory="stock" and region ="us"','select code from ticker_info where catagory="stock" and region ="hk"','select code from ticker_info where catagory="stock" and region ="cn"']
for sql in sql_list:

    result=mysql.read_query(sql)
    #print(len(result))
    code_str=''
    for code in result:
        code_str+=(code[0]+',')
    code_str=code_str[:-1]
    data=ths_hq_query(code_str,'2022-01-01','2022-03-31')
    sql='insert ignore into ticker_data (code,date,close_price,volume) values(%s,%s,%s,%s)'

val=[]

for i,d in data.iterrows():
        val.append((d['thscode'],d['time'],d['close'],d['volume']))

result=mysql.wirte_many_query(sql,val)
print(result)
#print(code_str)
def ths_hq_query(code_str,start_data_str,end_date_str):
    '''use ths_hq func for old data,no af/capital for hk/us'''
    
    data=THS_HQ(code_str,'close,volume','fill:Blank',start_data_str,end_date_str)
    if data.errorcode!=0:
        logger.error('ths query failed with errmsg %s,%s'%(data.errorcode,data.errmsg))
    else:
        
        
        data=data.data
        logger.info('got %i ths record(s)'%(data.shape[0]))
        data.replace({np.nan: None},inplace=True)
            
        return data
data=ths_hq_query(code_str,'2022-01-01','2022-03-31')

In [172]:
sql_list=['select code from ticker_info where catagory="stock" and region ="us"','select code from ticker_info where catagory="stock" and region ="hk"','select code from ticker_info where catagory="stock" and region ="cn"']
for sql in sql_list:

    result=mysql.read_query(sql)
    #print(len(result))
    code_str=''
    for code in result:
        code_str+=(code[0]+',')
    code_str=code_str[:-1]
    data=ths_hq_query(code_str,'2021-05-01','2021-06-30')
    sql='insert ignore into ticker_data (code,date,close_price,volume) values(%s,%s,%s,%s)'

    val=[]

    for i,d in data.iterrows():
            val.append((d['thscode'],d['time'],d['close'],d['volume']))

    result=mysql.wirte_many_query(sql,val)
    print(result)
#print(code_str)

05-25 15:45 logger       INFO     got a cursor
05-25 15:45 logger       INFO     got a cursor
05-25 15:45 logger       INFO     Accepted read sql query "select code from ticker_info where catagory="stock" and region ="us""
05-25 15:45 logger       INFO     Accepted read sql query "select code from ticker_info where catagory="stock" and region ="us""
05-25 15:46 logger       INFO     got 256119 ths record(s)
05-25 15:46 logger       INFO     got 256119 ths record(s)
05-25 15:46 logger       INFO     got a cursor
05-25 15:46 logger       INFO     got a cursor
05-25 15:46 logger       INFO     executed and commited sql "insert ignore into ticker_data (code,date,close_price,volume) values(%s,%s,%s,%s)"
05-25 15:46 logger       INFO     executed and commited sql "insert ignore into ticker_data (code,date,close_price,volume) values(%s,%s,%s,%s)"
05-25 15:46 logger       INFO     got a cursor
05-25 15:46 logger       INFO     got a cursor
05-25 15:46 logger       INFO     Accepted read sql qu

256119


05-25 15:47 logger       INFO     got 102092 ths record(s)
05-25 15:47 logger       INFO     got 102092 ths record(s)
05-25 15:47 logger       INFO     got a cursor
05-25 15:47 logger       INFO     got a cursor
05-25 15:47 logger       INFO     executed and commited sql "insert ignore into ticker_data (code,date,close_price,volume) values(%s,%s,%s,%s)"
05-25 15:47 logger       INFO     executed and commited sql "insert ignore into ticker_data (code,date,close_price,volume) values(%s,%s,%s,%s)"
05-25 15:47 logger       INFO     got a cursor
05-25 15:47 logger       INFO     got a cursor
05-25 15:47 logger       INFO     Accepted read sql query "select code from ticker_info where catagory="stock" and region ="cn""
05-25 15:47 logger       INFO     Accepted read sql query "select code from ticker_info where catagory="stock" and region ="cn""


102092


05-25 15:47 logger       INFO     got 171789 ths record(s)
05-25 15:47 logger       INFO     got 171789 ths record(s)
05-25 15:47 logger       INFO     got a cursor
05-25 15:47 logger       INFO     got a cursor
05-25 15:47 logger       INFO     executed and commited sql "insert ignore into ticker_data (code,date,close_price,volume) values(%s,%s,%s,%s)"
05-25 15:47 logger       INFO     executed and commited sql "insert ignore into ticker_data (code,date,close_price,volume) values(%s,%s,%s,%s)"


171789


In [ ]:
#result=[cndf,hkdf,usdf] notanymore
sql='insert ignore into ticker_data (code,date,close_price,volume) values(%s,%s,%s,%s)'

val=[]

for i,d in data.iterrows():
        val.append((d['thscode'],d['time'],d['close'],d['volume']))

result=mysql.wirte_many_query(sql,val)
print(result)

In [ ]:
def hq_job():
#200w once, query ths for 1month

In [ ]:
def ths_query(region,result,start_date_str=(date.today()-timedelta(1)).__format__('%Y-%m-%d'),end_date_str=date.today().__format__('%Y-%m-%d')):
    '''query ths db for 3 pools of stock, fill np.nan with None and return result list'''
    #result=[cnstr,hkstr,usstr] not anymore
    '''HQ func: 
    
    THS_HQ('430418.BJ','close,volume,totalCapital,adjustmentFactorBackward1','fill:Blank','2021-03-01','2022-05-20') ['time','close',''column','totalCapital','adjustmentFactorBackward1']
    no af/capital for hk/us    '''
    if region=='CN':
        data=THS_DS(result[0],'ths_af_stock;ths_close_price_stock;ths_vol_stock;ths_market_value_stock',';100;100;','Fill:Blank',start_date_str,end_date_str)
    elif region=="HK":
        data=THS_DS(result[1],'ths_af_hks;ths_close_price_hks;ths_vol_hks;ths_market_value_hks',';100;1;HKD','Fill:Blank',start_date_str,end_date_str)
    else :
        data=THS_DS(result[2],'ths_af_uss;ths_close_price_uss;ths_vol_uss;ths_market_value_uss',';100;;USD','Fill:Blank',start_date_str,end_date_str)
    if data.errorcode!=0:
        logger.error('ths query failed for region %s with errmsg %s,%s'%(region,data.errorcode,data.errmsg))
    else:
        
        
        data=data.data
        if data.shape ==(0,0):
            logger.info('got no data from ths for data: %s'%start_date_str)
            return False
        else:
            logger.info('got %i ths record(s) for region %s'%(data.shape[0],region))
            if region=='CN':
                data['ths_market_value_stock']/=100000000
            elif region=='HK':
                data['ths_market_value_hks']/=100000000
            else:
                data['ths_market_value_uss']/=100000000

            data.replace({np.nan: None},inplace=True)
            
            return data

In [ ]:
def write_ticker_data(region,result,mysql):
    #result=[cndf,hkdf,usdf] notanymore
    sql='insert  into ticker_data (code,date,close_price,market_value,volume,authority_factor) values(%s,%s,%s,%s,%s,%s)'
    
    val=[]
    
    if region=='CN':
        for i,d in result.iterrows():
            val.append((d['thscode'],d['time'],d['ths_close_price_stock'],d['ths_market_value_stock'],d['ths_vol_stock'],d['ths_af_stock']))
    elif region=="HK":
        for i,d in result.iterrows():
            val.append((d['thscode'],d['time'],d['ths_close_price_hks'],d['ths_market_value_hks'],d['ths_vol_hks'],d['ths_af_hks']))
    else:
        for i,d in result.iterrows():
            val.append((d['thscode'],d['time'],d['ths_close_price_uss'],d['ths_market_value_uss'],d['ths_vol_uss'],d['ths_af_uss']))
    
    result=mysql.wirte_many_query(sql,val)
    return result

In [ ]:
def metajob(region):#region='CN'HK'US'
    THS_iFinDLogin('lzxh0011','450503')
    host='localhost'
    user='Local_Editor'
    password='QuantumGalaxy'
    database='qgdbs'
    mysql=MYSQL(host,user,password,database)
    result=query_code(mysql)
    if region=='US':
        day=date.today()-timedelta(1)
    else:
        day=date.today()
    day=day.strftime('%Y-%m-%d')
    day='2022-04-01'
    #day2=day
    day2='2022-05-04'
    result=ths_query(region,result,day,day2)
    if isinstance(result,pd.DataFrame):
        sqlresult=write_ticker_data(region,result,mysql)
        mysql.close()
        THS_iFinDLogout()
        return result,sqlresult

In [ ]:
result,sqlresult=metajob(region="US")
sqlresult

In [ ]:
result

In [ ]:
r=write_ticker_data("CN",result,mysql)
r

In [ ]:
THS_DataStatistics()

In [ ]:
mysql.close()
THS_iFinDLogout()

In [ ]:
host='localhost'
user='Local_Editor'
password='QuantumGalaxy'
database='qgdbs'
mysql=MYSQL(host,user,password,database)



In [ ]:
data=ths_query('CN',result,'2022-05-22','2022-05-22')
if isinstance(data,pd.DataFrame):
    print('good')

In [ ]:
result

In [ ]:
for df in result:
    df.to_csv('505-520.csv')


In [ ]:
test=result[1]
test = test.replace({np.nan: None})
test

In [ ]:
sql='insert ignore into ticker_data (code,date,close_price,market_value,volume,authority_factor) values(%s,%s,%s,%s,%s,%s)'
val=[]
for i,d in result[1].iterrows():
        val.append((d['thscode'],d['time'],d['ths_close_price_hks'],d['ths_market_value_hks'],d['ths_vol_hks'],d['ths_af_hks']))
result=mysql.wirte_many_query(sql,val)

In [ ]:
result

In [ ]:
THS_DataStatistics()

In [ ]:
df

In [ ]:
host='localhost'
user='Local_Editor'
password='QuantumGalaxy'
database='qgdbs'
mysql=MYSQL(host,user,password,database)

In [ ]:
sql='insert ignore into ticker_data (code,date,close_price,market_value,volume,authority_factor) values(%s,%s,%s,%s,%s,%s)'
val=[]
for i,data1 in data.data.iterrows():
    val.append((data1['thscode'],data1['time'],data1['ths_close_price_stock'],data1['ths_market_value_stock'],data1['ths_vol_stock'],data1['ths_af_stock']))
result=mysql.wirte_many_query(sql,val)
#mysql.close()
result
